# 04 · Tasks 3 & 4 — speed

*Maximum SI-SDR per minute, and the best single epoch*

Neural Audio and Speech Processing — Day 2 / Application to Speech Enhancement
Homework assignment, R. Scheibler (2026-07-02) · dataset: Voicebank-DEMAND (16 kHz)

> **Task 3.** Find the best combination of parameters to reach maximum SNR in the
> shortest time possible.
> **Task 4.** Best performance in a single epoch.

Both tasks are about the *budget*, not the architecture. The levers available
without touching the model are:

| Lever | Effect |
|---|---|
| `lr` | the peak of the cosine schedule |
| `warmup_steps` | how long before the peak is reached — **the dominant term for a 1-epoch run** |
| `batch_size` | steps per epoch (smaller batch = more updates, less parallelism) |
| `remix` | augmentation: re-pairs clean speech with a shuffled noise inside the batch |
| activation | from notebook 02 |

### The warmup trap

One epoch is `16 904 // batch_size` steps ≈ **528 steps at batch 32**. The stock
schedule warms up over **500** steps — so in a single-epoch run the learning rate
is still ramping when the run ends and the model never trains at its peak rate.
`train.py` now reads `warmup_steps` from the config (and caps it at half the run as
a safety net), which makes this measurable.

In [ ]:
# --- Google Colab bootstrap (does nothing when running locally) --------------
# IMPORTANT: point this at the fork that contains the homework modifications
# (models/crn.py with a selectable activation, train.py with --warmup-steps).
REPO_URL = "https://github.com/Ahmed-AlGhosaini/nanoSE.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os

    if not os.path.exists("nanoSE"):
        !git clone -q $REPO_URL nanoSE
    %cd nanoSE
    !pip install -q -r requirements.txt

    import torch
    if not torch.cuda.is_available():
        print("No GPU! Runtime > Change runtime type > T4 GPU, then re-run this cell.")

In [ ]:
import sys
from pathlib import Path

# Make notebooks/nb_utils.py importable no matter where the kernel was started
for candidate in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent / "notebooks"):
    if (candidate / "nb_utils.py").exists():
        sys.path.insert(0, str(candidate))
        break

import matplotlib.pyplot as plt
import pandas as pd

import nb_utils

ROOT = nb_utils.bootstrap()   # chdir to the repository root + print the device

In [ ]:
best_lr = nb_utils.registry().get("best_lr_value", 1e-3)
best_activation = nb_utils.registry().get("best_activation_name", "leaky_relu")
try:
    best_lr = float(best_lr)
except (TypeError, ValueError):
    best_lr = 1e-3
print(f"carried over from the earlier notebooks: lr={best_lr:g}, activation={best_activation}")

from dataset import VoiceBankDemandDataset
n_segments = len(VoiceBankDemandDataset(split="train"))
for batch_size in (16, 32, 64):
    print(f"batch {batch_size:>3}: {n_segments // batch_size:>5} steps per epoch")

## Task 4 — the best single epoch

Six one-epoch runs. Everything not named in the table is the baseline value.

> **Resuming after a disconnect.** Every finished run is recorded in
> `notebooks/experiment_runs.json`, and the sweep loops skip anything already recorded.
> Re-running the cell after a Colab timeout continues where it stopped instead of
> starting over.

In [ ]:
single_epoch_variants = {
    "stock schedule":            dict(),
    "warmup=50":        dict(warmup_steps=50),
    "warmup=50 lr=3e-3": dict(warmup_steps=50, lr=3e-3),
    "warmup=50 lr=3e-3 batch=16": dict(warmup_steps=50, lr=3e-3, batch_size=16),
    "warmup=50 lr=3e-3 batch=64": dict(warmup_steps=50, lr=3e-3, batch_size=64),
    "warmup=50 lr=3e-3 remix": dict(warmup_steps=50, lr=3e-3, remix=True),
}

single_runs = {}
for label, overrides in single_epoch_variants.items():
    slug = label.replace(" ", "_").replace("=", "").replace("-", "")
    key = f"1ep_{slug}"
    done = nb_utils.recall(key)
    if done is not None:
        single_runs[label] = done
        print(f"[skip] {label:<28} -> {done}")
        continue

    config = nb_utils.write_config(
        f"exp_{key}.py",
        name=key,
        model=f'CRNTiny(activation="{best_activation}")',
        docstring=f"Task 4: single-epoch variant -- {label}.",
        epochs=1,
        **overrides,
    )
    single_runs[label] = nb_utils.remember(key, nb_utils.run_training(config))

In [ ]:
labels = list(single_runs)
runs = [single_runs[label] for label in labels]

table = nb_utils.summarize(runs, labels=labels, best_epoch=True)
columns = ["label", "val_si_sdr", "pesq", "estoi", "dnsmos"]
if "time_min" in table:
    columns.append("time_min")
table[columns].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
nb_utils.plot_bars(list(table.label), list(table.val_si_sdr),
                   title="SI-SDR after exactly one epoch",
                   ylabel="Validation SI-SDR (dB)",
                   baseline=float(table.loc[table.label == "stock schedule", "val_si_sdr"].iloc[0]), ax=ax)
plt.tight_layout()
plt.show()

winner = table.iloc[0]
stock = float(table.loc[table.label == "stock schedule", "val_si_sdr"].iloc[0])
print(f"best single epoch: {winner.label} -> {winner.val_si_sdr:.2f} dB "
      f"({winner.val_si_sdr - stock:+.2f} dB vs the stock schedule)")

## Task 3 — maximum SI-SDR in the shortest time

Same question, longer horizon: plotted against **wall-clock minutes** instead of
epochs, which is the axis the task actually asks about. `nb_utils` records the
per-epoch time reported by `train.py` in `progress.json`, so any run trained
through these notebooks can be placed on this axis.

Three candidate recipes at `SPEED_EPOCHS` epochs each, compared against the
baseline curve.

In [ ]:
SPEED_EPOCHS = 5

speed_variants = {
    "baseline recipe":  dict(),
    "fast: short warmup + high lr": dict(warmup_steps=100, lr=max(best_lr, 3e-3)),
    "fast + batch 64":  dict(warmup_steps=100, lr=max(best_lr, 3e-3), batch_size=64),
    "fast + remix":     dict(warmup_steps=100, lr=max(best_lr, 3e-3), remix=True),
}

speed_runs = {}
for label, overrides in speed_variants.items():
    slug = label.replace(" ", "_").replace(":", "").replace("+", "").replace("=", "")
    key = f"speed_{slug}"
    done = nb_utils.recall(key)
    if done is not None:
        speed_runs[label] = done
        print(f"[skip] {label:<30} -> {done}")
        continue

    config = nb_utils.write_config(
        f"exp_{key}.py",
        name=key,
        model=f'CRNTiny(activation="{best_activation}")',
        docstring=f"Task 3: time-to-quality variant -- {label}.",
        epochs=SPEED_EPOCHS,
        **overrides,
    )
    speed_runs[label] = nb_utils.remember(key, nb_utils.run_training(config))

In [ ]:
labels = list(speed_runs)
runs = [speed_runs[label] for label in labels]

fig, ax = plt.subplots(figsize=(9, 5))
nb_utils.plot_curves(runs, labels=labels, metric="val_si_sdr", x="cumulative_time_min", ax=ax,
                     title="Validation SI-SDR against wall-clock training time")
plt.tight_layout()
plt.show()

In [ ]:
# "Time to reach X dB" is the number Task 3 is really asking for
baseline_run = nb_utils.recall("baseline")
baseline_history = nb_utils.load_metrics(baseline_run) if baseline_run else pd.DataFrame()

if not baseline_history.empty:
    trained = baseline_history[baseline_history.epoch > 0]
    targets = [round(float(trained.val_si_sdr.max()) * f, 1) for f in (0.8, 0.9, 0.95)]
else:
    targets = [8.0, 9.0, 10.0]

rows = []
for label, run in speed_runs.items():
    row = {"recipe": label}
    for target in targets:
        row[f"min to {target} dB"] = round(nb_utils.time_to_reach(run, target), 1)
    rows.append(row)
if baseline_run is not None:
    row = {"recipe": "baseline (full run)"}
    for target in targets:
        row[f"min to {target} dB"] = round(nb_utils.time_to_reach(baseline_run, target), 1)
    rows.append(row)

pd.DataFrame(rows)

In [ ]:
# Cost side of the trade-off: seconds per epoch for each recipe
rows = []
for label, run in {**speed_runs, "baseline": baseline_run}.items():
    if run is None:
        continue
    history = nb_utils.load_metrics(run)
    if "time" not in history:
        continue
    epochs = history[history.epoch > 0]
    config = nb_utils.load_config(run)
    rows.append({
        "recipe": label,
        "batch_size": config.get("batch_size"),
        "lr": config.get("lr"),
        "remix": config.get("remix"),
        "s / epoch": round(float(epochs["time"].mean()), 1),
    })
pd.DataFrame(rows)

## Discussion

*Fill in with your numbers.* Expected structure of the answer:

* **Task 4 (one epoch).** The warmup length dominates: with the stock 500-step
  warmup the model spends its only epoch at a fraction of the nominal learning
  rate. Shortening the warmup and raising the peak `lr` is worth far more in a
  1-epoch budget than any architectural change. A smaller batch buys more updates
  per epoch, but each is noisier — report which side wins here.
* **Task 3 (time to quality).** Note that the *y*-axis crossing time, not the final
  value, is the metric. `remix` costs nothing per step but changes the data
  distribution; a bigger batch is cheaper per segment on a GPU but takes fewer
  optimisation steps per epoch.
* **Validation is part of the wall clock.** Roughly a minute per epoch goes to
  PESQ/DNSMOS on CPU. For a pure speed run one would validate less often — worth
  mentioning as a caveat, since it inflates every number in the table equally.

**Next:** `05_final_experiment_and_report.ipynb` (Task 5 + the report).